# Session 4 · Data Cleaning
**Pandas & Data Manipulation** · Albert School × Politecnico di Milano

| Part | What | Time |
|---|---|---|
| 1 | Toolbox: every method of the slides on tiny tables, ready to reuse in Part 2 | 20 min |
| 2 | 30 questions on the NYC taxi data (10 imputation · 10 outliers · 10 duplicates) | 90 min |

**Rules for Part 2**: never modify `df`, work on copies; end every answer with a `#` comment giving the number in words.
Difficulty: ★ one method call · ★★ two or three steps · ★★★ a short pipeline.

In [ ]:
# Run once if a library is missing:
# !pip install scikit-learn recordlinkage      # recordlinkage also installs jellyfish

import numpy as np
import pandas as pd
import jellyfish
import recordlinkage
from sklearn.impute import KNNImputer
from sklearn.neighbors import LocalOutlierFactor

pd.set_option("display.max_columns", 20)

---
# Part 1 · Toolbox
Every method of the slides in its simplest form, on tiny tables that use the **same column names as the taxi data**.
Run each cell and look at the result. In Part 2, find the method you need here, copy its cell, and change the table and the column names.

## 1.1 Missing values

In [ ]:
mini = pd.DataFrame({
    "payment":    ["cash", "card", "card", "unknown", "cash", "card", None, "card"],
    "zone":       ["Midtown", None, "SoHo", "Midtown", None, "SoHo", "Midtown", "SoHo"],
    "passengers": [1, 2, None, 1, 1, None, 3, 1],
    "distance":   [1.2, 3.5, None, 0.8, 15.0, 2.1, None, 1.0],
    "fare":       [6.5, 14.0, 9.0, 5.0, 52.0, 10.0, 30.0, 5.5],
    "tip":        [0.0, None, 2.0, 1.0, None, 2.5, 0.0, 1.5],
    "total":      [6.5, 17.0, None, 6.0, 52.0, 12.5, 30.0, None],
})
mini

**Count the gaps**

In [ ]:
print(mini.isna().sum())                        # missing values in each column
print(mini.isna().any(axis=1).sum())            # rows with at least one gap
print(mini.isna().any(axis=1).mean() * 100)     # the same, as a percentage

**Hidden gaps**: a text such as `'unknown'` is a missing value in disguise

In [ ]:
pay = mini["payment"].replace("unknown", np.nan)
pay.isna().sum()

**Fill text with a constant**

In [ ]:
text_filled = mini[["payment", "zone"]].fillna("Unknown")
(text_filled == "Unknown").sum()                # cells filled in each column

**Drop** the rows that have a gap in some columns

In [ ]:
kept = mini.dropna(subset=["zone", "distance"])
print(len(mini), len(kept))
print((1 - len(kept) / len(mini)) * 100)        # % of rows lost

**Fill numbers with the mean, the median or the mode**

In [ ]:
print(mini["distance"].mean(), mini["distance"].median())   # mean far above median: skewed, use the median
dist_filled = mini["distance"].fillna(mini["distance"].median())

mode = mini["passengers"].mode()[0]             # the most frequent value
pax_filled = mini["passengers"].fillna(mode)

print(mini["distance"].std(), dist_filled.std())   # filling with one value shrinks the spread

**Ordered data**: repeat the last value (`ffill`) or draw a straight line (`interpolate`)

In [ ]:
temp = pd.Series([12, 13, None, None, 16])
pd.DataFrame({"temp": temp, "ffill": temp.ffill(), "interpolate": temp.interpolate()})

**Deduce** a value from the other columns when a rule exists: total = fare + tip

In [ ]:
mini["total"].fillna(mini["fare"] + mini["tip"])

**KNN imputation**

In [ ]:
X = mini[["distance", "fare"]]
filled = pd.DataFrame(KNNImputer(n_neighbors=2).fit_transform(X), columns=X.columns, index=X.index)
# fit_transform returns a NumPy array: pd.DataFrame(...) turns it back into a table

missing = mini["distance"].isna()               # the rows that had no distance
comp = mini.loc[missing, ["fare"]]
comp["knn"] = filled.loc[missing, "distance"]
comp["median"] = mini["distance"].median()
comp

## 1.2 Outliers

In [ ]:
fares = pd.Series([7.5, 9.0, 6.5, -8.0, 12.0, 950.0, 10.5, 8.0, 45.0, 11.0,
                   9.5, 7.0, 13.5, 8.5, 10.0, 6.0, 14.0, 9.0, 11.5, 8.0])

**Look first** → `describe()` gives the minimum, the quartiles and the maximum

In [ ]:
fares.describe()

**Correct what you can explain** → `s.where(condition, other)` keeps the value where the condition is True and puts `other` elsewhere (NaN if you give no `other`).

In [ ]:
print((fares < 0).sum())                              # negative fares
fares1 = fares.abs()                                  # wrong sign: take the absolute value
print(fares1.min())

print((fares1 > 500).sum())                           # fares above 500: the decimal point was lost
fares2 = fares1.where(fares1 <= 500, fares1 / 100)    # keep if <= 500, otherwise divide by 100
print(fares2.max())

pax = pd.Series([1, 2, 0, 1, 9, None, 3])
pax_ok = pax.where(pax.between(1, 6))                 # impossible values become NaN
print(pax.isna().sum(), pax_ok.isna().sum())          # NaN before and after

**z-score** → how many standard deviations a value is from the mean

In [ ]:
z1 = (fares1 - fares1.mean()) / fares1.std()
z2 = (fares2 - fares2.mean()) / fares2.std()
print((z1.abs() > 3).sum(), (z2.abs() > 3).sum())
print(fares1[z1.abs() > 3].tolist())                  # only the 950 typo
print(fares2[z2.abs() > 3].tolist())                  # once 950 is fixed, the 45 shows up: the typo was hiding it

**IQR fences** → the boxplot rule. Then look at the largest values: errors or real?

In [ ]:
q1, q3 = fares2.quantile([0.25, 0.75])
iqr = q3 - q1
low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(low, high)                                      # the two fences
print(((fares2 < low) | (fares2 > high)).sum())       # values outside
fares2.sort_values(ascending=False).head(3)           # the 3 largest values

**Cap** the extreme values

In [ ]:
hi = fares2.quantile(0.95)
capped = fares2.clip(upper=hi)                        # values above hi become hi
print(hi, (fares2 > hi).sum(), capped.max())          # limit, values changed, new maximum

**Odd pairs** → two values that are normal on their own can be impossible together. Combine two conditions with `&`.

In [ ]:
pts = pd.DataFrame({"distance": [0.8, 1.0, 1.2, 1.5, 1.8, 2.0, 2.4, 2.8, 3.0, 3.5, 4.0, 3.0],
                    "fare":     [5.0, 5.5, 6.0, 7.0, 7.5, 8.0, 9.0, 10.0, 10.5, 12.0, 13.0, 40.0]})
pts[(pts["distance"] < 5) & (pts["fare"] > 30)]       # a short ride with a high fare

**LOF** finds the odd rows without writing a rule. Standardise the columns first (same scale), then `fit_predict` gives -1 for an outlier and 1 for a normal row.

In [ ]:
Z = (pts - pts.mean()) / pts.std()                    # standardise
pts["lof_label"] = LocalOutlierFactor(n_neighbors=3, contamination=0.05).fit_predict(Z)
print((pts["lof_label"] == -1).sum())                 # contamination=0.05: flag about 5% of the rows
pts[pts["lof_label"] == -1]

## 1.3 Duplicates
Each cell continues from the table `clean` of the previous one.

In [ ]:
trips = pd.DataFrame({
    "ride_id":     [1, 2, 2, 3, 4, 4, 5, 6],
    "pickup":      pd.to_datetime(["2019-03-01 08:00:05", "2019-03-01 08:03:10", "2019-03-01 08:03:10",
                                   "2019-03-01 08:10:00", "2019-03-01 08:15:30", "2019-03-01 08:15:30",
                                   "2019-03-01 08:15:42", "2019-03-01 08:30:00"]),
    "pickup_zone": ["Midtown East", "Gramercy", "Gramercy", "SoHo", "Midtown East", "Midtown East",
                    "Midtwon East", "Garmercy"],
    "payment":     ["cash", "credit card", "credit card", " Cash", "CREDIT CARD", "credit card",
                    "credit card", "cash"],
    "distance":    [1.2, 2.5, 2.5, 1.8, 5.1, 5.1, 5.1, 1.0],
    "fare":        [7.5, 12.0, 12.0, 9.0, 20.0, 20.0, 20.0, 6.5],
    "total":       [7.5, 15.0, 15.0, 9.0, 25.0, 23.0, 25.0, 6.5],
})
trips

**Exact duplicates**

In [ ]:
print(trips.duplicated().sum())
clean = trips.drop_duplicates()
clean.shape

**Same id, different values** → `keep=False` marks every copy, so you can see them side by side.

In [ ]:
print(clean.duplicated(subset="ride_id").sum())       # ids still repeated
clean[clean.duplicated(subset="ride_id", keep=False)].sort_values("ride_id")

In [ ]:
clean = clean.sort_values("total").drop_duplicates(subset="ride_id", keep="last")   # keep the higher total
clean.shape

**Normalise text**

In [ ]:
print(clean["payment"].nunique())
clean["payment"] = clean["payment"].str.strip().str.lower()    # .str.title() would give "Cash"
print(clean["payment"].nunique())
clean["payment"].value_counts()

**String similarity** with `jellyfish`

In [ ]:
pairs = [("Midtown East", "Midtwon East"), ("SoHo", "NoHo")]
for a, b in pairs:
    print(a, "|", b,
          jellyfish.levenshtein_distance(a, b),                 # edits needed: 0 = equal
          round(jellyfish.jaro_winkler_similarity(a, b), 2))    # from 0 to 1: 1 = equal
# SoHo and NoHo look alike but are two different places

**Values not in an official list**

In [ ]:
valid = ["Gramercy", "Midtown East", "SoHo"]
zone = clean["pickup_zone"].str.strip()
bad = zone[~zone.isin(valid)].dropna()                # a missing zone is not a typo
print(len(bad), bad.nunique())                        # rows, distinct misspellings
bad.value_counts()

**Candidate pairs** → comparing every row with every other is too slow; blocking compares only rows that share a value

In [ ]:
n = len(clean)
print(n * (n - 1) // 2)                               # every pair of rows

ix = recordlinkage.Index()
ix.block("pickup_zone")                               # only rows with the same zone
print(len(ix.index(clean)))

**Near duplicates** → ride 5 is ride 4 sent again a few seconds later, with a new id and a misspelled zone. Record linkage finds it in four steps: **pick pairs** (only rides picked up in the same minute), **score** each field (fare, distance, zone), **decide** (a match agrees on all three), **merge** (one golden record per pair).

In [ ]:
# Take the pickup time of every ride
t = clean["pickup"]
# Split it into three new columns: date, hour and minute (the seconds are left out)
clean["date"], clean["hour"], clean["minute"] = t.dt.date, t.dt.hour, t.dt.minute

# PICK PAIRS: create an indexer, the object that decides which rows to compare
ix = recordlinkage.Index()
# Pair only the rows with the same date, hour and minute (blocking)
ix.block(["date", "hour", "minute"])
# Build the candidate pairs: a MultiIndex of (row label, row label)
pairs = ix.index(clean)

# SCORE: create a comparer, the object that holds the field-by-field tests
c = recordlinkage.Compare()
# 1 if the two fares are equal, 0 otherwise
c.exact("fare", "fare", label="fare")
# 1 if the two distances are equal, 0 otherwise
c.exact("distance", "distance", label="distance")
# 1 if the two zones are similar (Jaro-Winkler >= 0.7), 0 otherwise: typos and lower case still count as equal
c.string("pickup_zone", "pickup_zone", method="jarowinkler", threshold=0.7, label="zone")
# Run the tests: one row per pair, one 0/1 column per test
f = c.compute(pairs, clean)

# DECIDE: a pair is a match when it agrees on all 3 fields
matches = f[f.sum(axis=1) == 3]
matches

In [ ]:
# In each pair, level 1 is the original row (smaller label), level 0 the copy sent later
orig = matches.index.get_level_values(1)
copy = matches.index.get_level_values(0)
# MERGE: one golden record per pair. Start from the original row and fill its gaps (NaN) with the copy's values
golden = clean.loc[orig].combine_first(clean.loc[copy].set_axis(orig))[clean.columns]
# Rebuild clean: remove both rows of every pair, then put the golden records back
clean = pd.concat([clean.drop(index=orig.append(copy)), golden]).sort_index()
clean.shape

---
# Part 2 · The taxi data
Same rides as Sessions 2–3, with more problems. Two files in `data/`:

| file | rows | what |
|---|---|---|
| `taxis_dirty.csv` | 6,508 | one row per ride (15 columns, `ride_id` is the key) |
| `zones.csv` | 213 | official zone names and their borough |

In [ ]:
df = pd.read_csv("data/taxis_dirty.csv", parse_dates=["pickup", "dropoff"])
print(df.shape)
df.head()

## A · Imputation (Q1–Q10)

**Q1 ★** How many missing values does each column have? Which column has the most?

In [ ]:
# your code here


**Q2 ★** What percentage of rides has **at least one** missing value?

In [ ]:
# your code here


**Q3 ★** `payment` hides missing values as the text `"unknown"`. Replace them with `np.nan` (in a new Series `pay`) and count the missing payments.

In [ ]:
# your code here


**Q4 ★** Fill the missing `pickup_zone` and `dropoff_zone` with `"Unknown"` (in a new DataFrame `zones_filled`). How many cells were filled in each column?

In [ ]:
# your code here


**Q5 ★★** Drop the rides where `pickup_borough` **or** `dropoff_borough` is missing. How many rides remain, and what percentage did you lose?

In [ ]:
# your code here


**Q6 ★★** Fill the missing `passengers` with the **mode**. Which value is it, and how many rides did you fill?

In [ ]:
# your code here


**Q7 ★★** Compare the mean and the median of `distance`. Which one would you use to fill its missing values, and why? Fill them (new Series `dist_filled`).

In [ ]:
# your code here


**Q8 ★★** Fill `distance` with the **mean** and with the **median**. Compare the standard deviation of the original column with both filled versions. What happens?

In [ ]:
# your code here


**Q9 ★★** Fill the missing `distance` values with **KNN**. Work on `X = df[["distance", "fare", "total"]]`: `KNNImputer(n_neighbors=5)` fills each missing distance with the mean distance of the 5 rides with the closest `fare` and `total`. `fit_transform` returns a NumPy array: put it back into a DataFrame `filled` with `pd.DataFrame(..., columns=X.columns, index=X.index)`. How many distances were filled, and what is their mean?

In [ ]:
# your code here


**Q10 ★★** Compare the KNN fill (Q9) with the median fill (Q7, 1.65 miles). For the rides whose distance was missing, build a table with `fare`, the KNN distance and the median distance, and show the 5 rides with the highest fare. Which fill is more believable, and why?

In [ ]:
# your code here


## B · Outliers (Q11–Q20)

**Q11 ★** Run `describe()` on `fare`, `distance` and `tip`. What is the largest fare?

In [ ]:
# your code here


**Q12 ★** How many fares are negative? Correct them with `.abs()` (new Series `fare`). What is the new minimum?

In [ ]:
# your code here


**Q13 ★** A taxi carries 1 to 6 passengers. Set every other value to NaN (new Series `passengers`, use `.where`). How many NaN are there before and after?

In [ ]:
# your code here


**Q14 ★** Fares above 500 are typing errors (950 should be 9.50). How many are there? Divide them by 100 (new Series `fare2`, from `fare`). What is the new maximum?

In [ ]:
# your code here


**Q15 ★★** Compute the z-score of `fare` and of `fare2`. How many rides have |z| > 3 in each case? Why is the number so different?

In [ ]:
# your code here


**Q16 ★★** Apply the IQR rule to `fare2`. What are the two fences, and how many rides fall outside?

In [ ]:
# your code here


**Q17 ★★** Apply the IQR rule to `distance`: how many rides fall outside? Then show the 5 longest rides (`ride_id`, `distance`, `fare`, `pickup_zone`, `dropoff_zone`). Are they errors?

In [ ]:
# your code here


**Q18 ★** Cap `tip` at its 99th percentile with `.clip(upper=...)` (new Series `tip_capped`). What is the limit, how many tips were lowered, and what is the new maximum?

In [ ]:
# your code here


**Q19 ★★** A distance of 0 is possible (cancelled ride) and a fare above 50 is possible (long ride), but not both together. How many rides have `distance == 0` **and** `fare2 > 50`? Show them.

In [ ]:
# your code here


**Q20 ★★★** Let LOF find the odd rides without a rule:
1. `rides = df[["ride_id", "distance"]].copy()`, add the column `fare` = `fare2`, and drop the rows with NaN.
2. Standardise `rides[["distance", "fare"]]` into `Z`.
3. `LocalOutlierFactor(n_neighbors=50, contamination=0.01).fit_predict(Z)` into a column `lof_label`.

How many rides are flagged (label -1)? Show the first 5.

In [ ]:
# your code here


## C · Duplicates (Q21–Q30)
Each question continues from the table `d` of the previous one.

**Q21 ★** How many rows are exact duplicates? Drop them into a new table `d`. What is its shape?

In [ ]:
# your code here


**Q22 ★** In `d`, how many `ride_id` values are still repeated? Show all copies side by side, sorted by `ride_id`.

In [ ]:
# your code here


**Q23 ★★** For each repeated ride keep the copy with the **higher total** (sort first, then `drop_duplicates`). How many rows does `d` have now?

In [ ]:
# your code here


**Q24 ★** Normalise `payment` with `.str.strip().str.lower()`. How many distinct values are there before and after?

In [ ]:
# your code here


**Q25 ★** Normalise `pickup_borough` with `.str.strip().str.title()`. How many distinct values are there before and after? How many rides per borough?

In [ ]:
# your code here


**Q26 ★** Compute the Levenshtein distance and the Jaro-Winkler similarity (`jellyfish`) for these pairs. Which pair is a **trap** (similar strings, different zones)?
`("Gramercy", "Gramecy")`, `("JFK Airport", "JKF Airport")`, `("Midtown East", "Midtown West")`

In [ ]:
# your code here


**Q27 ★★** Load the official zone names: `valid = pd.read_csv("data/zones.csv")["zone"].tolist()`. How many `pickup_zone` values in `d` (after `.str.strip()`) are **not** in `valid`? How many distinct misspellings? Show the most frequent ones.

In [ ]:
# your code here


**Q28 ★★** To find duplicates that are not exact, we compare pairs of rows. How many pairs would we compare (a) with every pair, using the formula n(n-1)/2, (b) blocking on `dropoff_zone` (`recordlinkage.Index()`)?

In [ ]:
# your code here


**Q29 ★★** Some rides were sent twice by a second device: new `ride_id`, pickup a few seconds later (same minute), `pickup_zone` misspelled. Add the columns `date`, `hour` and `minute` of `pickup` (`.dt.date`, `.dt.hour`, `.dt.minute`) and count the duplicates on these three columns, `fare`, `distance` and `dropoff_zone`. Why not use `ride_id` or `pickup_zone`?

In [ ]:
# your code here


**Q30 ★** Drop these near duplicates. How many rides are left? Compare with the 6,433 rides of the original seaborn file.

In [ ]:
# your code here


---
**Done.** Write three lines on the cleaning decision you found hardest, and why.